# mutation_burden.ipynb
*These analyses have been replaced by regressions in `logistic_regression.ipynb` but are preserved for archival reasons*
- Do ecDNA+ tumors have greater somatic mutation burden (not accounting for tumor type)?
- Do ecDNA+ tumors have greater likely pathogenic somatic mutation burden (same)?*
- Do ecDNA+ tumors have greater pathogenic somatic mutation burden (same)?**

\* Impact HIGH or MODERATE, Polyphen + SIFT deleterious if available, and within a cancer gene.  
** (PBTA only) pathogenicity from HotSpot annotations.***  
*** Hotspots described in  
    https://github.com/kids-first/kf-annotation-tools/blob/master/docs/SOMATIC_SNV_ANNOT_README.md#source-specific-inputs  
    https://www.cancerhotspots.org/#/about
    

We compared variant count and stratified ecDNA by tumor type.

## RESULTS  

Biosample table with mutation burdens saved to `out/biosmple_mutation_burden.tsv`.  
Plots saved to `out/*mutation_burden*.svg`.

### Mann-Whitney U test p-values, all tumors:  
|         |  all |    lp|     p|
|---------|------|------|------|
|ec vs na |1e-16*|1e-13*| 7e-4*|
|ec vs chr|  0.80|  0.07|  0.17|
|chr vs na|1e-14*|1e-16*| 1e-6*|

all, lp: n=1798; tumor types = ['CPT' 'EMBT' 'EPN' 'ETMR' 'GCT' 'GNT' 'HGG' 'LGG' 'MBL' 'NBL' 'OS' 'PINT'
 'PNST' 'RMS' 'SARC']  
p: n=978; tumor types = ['CPT' 'EMBT' 'EPN' 'GCT' 'GNT' 'HGG' 'MBL' 'NBL' 'PINT' 'PNST' 'SARC']

### MWU test p-values, MBL tumors:
|         |  all |   lp|    p|
|---------|------|-----|-----|
|ec vs na |0.003*| 0.22| 0.14|
|ec vs chr|  0.79| 0.06| 0.17|
|chr vs na|  0.16| 0.16| 0.57|

all, lp: n=272  
p: n=221  

### MWU test p-values, HGG tumors:
|         |  all |    lp|    p|
|---------|------|------|-----|
|ec vs na | 6e-4*|  0.10| 0.36|
|ec vs chr|  0.35|  0.32| 0.10|
|chr vs na| 0.02*|0.005*| 0.36|

all, lp: n=270  
p: n=224  

## Data import
Parse SNV data into a DataFrame indexed by biosample with columns: 
| biosample_id | cancer_type | amplicon_class | all_counts | likely_pathogenic_counts | pathogenic_counts |
|--------------|-------------|----------------|------------|--------------------------|-------------------|
| `str`        | `str`       | `str`          | `int`      | `int`                    | `dbl`             | 

In [ ]:
import pandas as pd
import os
from pathlib import Path
import scipy
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('../../src')
from data_imports import *
from somatic_snv_functions import *

OUT_DIR=Path('out')
OUT_DIR.mkdir(parents=True,exist_ok=True)

In [ ]:
df = merge_ecDNA_counts(import_biosamples(),
                   reindex_counts(read_manifest(),get_variant_count_df())
                  )
# write df, including duplicates, then drop duplicates
path=OUT_DIR/'biosample_mutation_burden.tsv'
#df.to_csv(path,sep='\t')

df = df[df.in_snv_set]
print(f'lp n={df.shape[0]}')
print(f' p n={df[~df.pathogenic_counts.isna()].shape[0]}')
df.head(n=10)

In [ ]:
df['amplicon_class'].value_counts()

# Tests and plots
Compare mutation burden across amplicon classes by Wilcoxon rank sum test,
make boxplots

In [ ]:
def compare_somatic_mutation_burden(df, burden_colname, log=True, test=scipy.stats.mannwhitneyu):
    # parameter parsing
    if log:
        df['y_counts'] = np.log1p(df[burden_colname])
        ylab = 'Log (mutation burden)'
    else:
        df['y_counts'] = df[burden_colname]
        ylab = 'mutation burden'

    # subset tumor types, valid examples
    df = df.dropna(axis=0,subset=['y_counts'])
    target_types = get_target_tumor_types(df)
    df = df[df["cancer_type"].isin(target_types)].copy()
    print(target_types)
    print(f'n={df.shape[0]}')
    
    # plot
    ax = sns.violinplot(
        data = df,
        x = 'amplicon_class',
        y = 'y_counts',
        order = ['ecDNA','chromosomal','no amplification',]
    )

    # figure post-formatting
    sns.despine()
    ax.set_ylabel(ylab)
    ax.set_xlabel(None)
    ax.set_ylim(bottom=0)
    cts = df["amplicon_class"].value_counts().to_dict()
    new_labels = [
        f"{label.get_text()}\n(n={cts[label.get_text()]})"
        for label in ax.get_xticklabels()
    ]
    ax.set_xticklabels(new_labels)

    # statistics
    classes = sorted(list(df.amplicon_class.unique())) # ['ecDNA','intrachromosomal','no amplification']
    for i in range(len(classes)):
        for j in range(i+1,len(classes)):
            print("Comparing",classes[i],"to",classes[j])
            a = df[df.amplicon_class == classes[i]]['y_counts']
            b = df[df.amplicon_class == classes[j]]['y_counts']
            print(test(a,b))
    return ax

In [ ]:
ax = compare_somatic_mutation_burden(df,'all_counts')

plt.savefig('out/mutation_burden_all.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df,'likely_pathogenic_counts')
ax.set_ylabel('Log (likely pathogenic mutation burden)')
plt.savefig('out/mutation_burden_lp.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df,'pathogenic_counts')
ax.set_ylabel('Log (pathogenic mutation burden)')
plt.savefig('out/mutation_burden_p.svg')
#plt.show()

In [ ]:
def stratify_ecDNA_by_tumor_type(df):
    df = df.loc[:,['amplicon_class','cancer_type']]
    df = df.groupby(['cancer_type','amplicon_class']).size()
    return df

## Stratifications by tumor type

In [ ]:
strat = stratify_ecDNA_by_tumor_type(df)
strat

In [ ]:
df_MBL = df[df['cancer_type'].isin(['MBL'])].copy()
df_HGG = df[df['cancer_type'].isin(['HGG'])].copy()

In [ ]:
ax = compare_somatic_mutation_burden(df_MBL,'all_counts')
plt.savefig('out/MBL_mutation_burden_all.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df_MBL,'likely_pathogenic_counts')
ax.set_ylabel('Log (likely pathogenic mutation burden)')
plt.savefig('out/MBL_mutation_burden_lp.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df_MBL,'pathogenic_counts')
ax.set_ylabel('Log (pathogenic mutation burden)')
plt.savefig('out/MBL_mutation_burden_p.svg')
#plt.show()

In [ ]:
compare_somatic_mutation_burden(df_HGG,'all_counts')
plt.savefig('out/HGG_mutation_burden_all.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df_HGG,'likely_pathogenic_counts')
ax.set_ylabel('Log (likely pathogenic mutation burden)')
plt.savefig('out/HGG_mutation_burden_lp.svg')
#plt.show()

In [ ]:
ax = compare_somatic_mutation_burden(df_HGG,'pathogenic_counts')
ax.set_ylabel('Log (pathogenic mutation burden)')
plt.savefig('out/HGG_mutation_burden_p.svg')
#plt.show()